In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

train_metadata = pd.read_csv("./datasets/task1_data/train_metadata.csv")
hog_pca = pd.read_csv("./datasets/task1_data/hog_pca.csv")

full_train_df = pd.merge(left=hog_pca, right=train_metadata, on="image_id", how="inner")


def calculate_distances(image, training_images):
    distances = euclidean_distances([image], training_images)
    return distances.flatten()


def do_knn(image, training_images, k):
    distances = calculate_distances(image, training_images)
    sorted_distance_indices = np.argsort(distances)
    k_most_similar_indices = sorted_distance_indices[:k]
    labels = [
        full_train_df.iloc[index]["class_name"] for index in k_most_similar_indices
    ]
    label_vote_counts = Counter(labels)
    top_votes = label_vote_counts.most_common(2)
    if len(top_votes) > 1 and top_votes[0][1] == top_votes[1][1]:
        return do_knn(image, training_images, k + 1)
    return top_votes[0][0]


K = 5

training_features = full_train_df.drop(
    columns=["image_id", "class_name", "class_id", "image_path"]
)

test_image_features = training_features.iloc[0]

# --- real train test split
train_df, val_df = train_test_split(training_features, test_size=0.2, random_state=2718)
X_train = train_df.drop(columns=["image_id", "class_name", "class_id"])
y_train = train_df["class_name"]

X_val = val_df.drop(columns=["image_id", "class_name", "class_id"])
y_val = val_df["class_name"]

correct_preds = 0
total_tests = len(X_val)


prediction = do_knn(test_image_features, training_features, k=K)
true_label = full_train_df.iloc[0]["class_name"]
print(f"The model predicted {prediction}")
print(f"The true label was {true_label}")

The model predicted spider
The true label was bird
